<a href="https://colab.research.google.com/github/korallisampath-collab/Sampath/blob/main/FILE_RECOMMENDATIONS_AGENTIC_AI_1ST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PROJECT 5
# AGENTIC AI-BASED INTELLIGENT FILE RECOMMENDATION SYSTEM
# ============================================================
#
# COMPLETE GOOGLE COLAB VERSION
# STEP 1 -> STEP 12
#
# FEATURES
# ------------------------------------------------------------
# 1. Project setup
# 2. Automatic ZIP/document repository
# 3. PDF/DOCX/TXT/PPTX/HTML extraction
# 4. Intelligent overlapping chunking
# 5. Transformer embeddings
# 6. Vector database
# 7. Hybrid semantic + keyword search
# 8. Agentic AI query planning
# 9. Intelligent file ranking
# 10. RAG + evidence
# 11. Gradio interface
# 12. Evaluation and testing
#
# IMPORTANT:
# This code does NOT repeatedly ask for upload.
# ============================================================

# ============================================================
# 0. BASIC IMPORTS
# ============================================================

import os
import sys
import re
import json
import time
import shutil
import zipfile
import hashlib
import subprocess
import warnings

from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

print("=" * 80)
print("PROJECT 5")
print("AGENTIC AI-BASED INTELLIGENT FILE RECOMMENDATION SYSTEM")
print("COMPLETE EXECUTION")
print("=" * 80)

# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

print("\n")
print("=" * 80)
print("SYSTEM SETUP")
print("=" * 80)

packages = [
    ("numpy", "numpy"),
    ("pypdf", "pypdf"),
    ("python-docx", "docx"),
    ("python-pptx", "pptx"),
    ("sentence-transformers", "sentence_transformers"),
    ("gradio", "gradio"),
    ("beautifulsoup4", "bs4")
]

for package_name, import_name in packages:

    try:

        __import__(import_name)

        print(
            f"✓ {package_name} already installed"
        )

    except ImportError:

        print(
            f"Installing {package_name}..."
        )

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

        print(
            f"✓ {package_name} installed"
        )

import numpy as np

from pypdf import PdfReader
from docx import Document
from pptx import Presentation
from bs4 import BeautifulSoup

from sentence_transformers import SentenceTransformer

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_NAME = "Agentic_File_Recommendation"

BASE_DIR = Path(
    "/content"
) / PROJECT_NAME

DATA_DIR = BASE_DIR / "data"

DOCUMENTS_DIR = (
    DATA_DIR / "documents"
)

PROCESSED_DIR = (
    DATA_DIR / "processed"
)

CHUNKS_DIR = (
    DATA_DIR / "chunks"
)

EMBEDDINGS_DIR = (
    BASE_DIR / "embeddings"
)

VECTOR_DB_DIR = (
    BASE_DIR / "vector_db"
)

RESULTS_DIR = (
    BASE_DIR / "results"
)

APP_DIR = (
    BASE_DIR / "app"
)

SRC_DIR = (
    BASE_DIR / "src"
)

ALL_DIRECTORIES = [
    DOCUMENTS_DIR,
    PROCESSED_DIR,
    CHUNKS_DIR,
    EMBEDDINGS_DIR,
    VECTOR_DB_DIR,
    RESULTS_DIR,
    APP_DIR,
    SRC_DIR
]

for directory in ALL_DIRECTORIES:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".docx",
    ".txt",
    ".pptx",
    ".html",
    ".htm"
}

# ============================================================
# STEP 1 - PROJECT SETUP
# ============================================================

print("\n")
print("=" * 80)
print("STEP 1 - PROJECT SETUP")
print("=" * 80)

print(
    "\nProject directory:"
)

print(
    BASE_DIR
)

print(
    "\nCreated project folders:"
)

for directory in ALL_DIRECTORIES:

    print(
        "  ✓",
        directory
    )

README_FILE = (
    BASE_DIR / "README.md"
)

README_FILE.write_text(
    """
# Agentic AI-Based Intelligent File Recommendation System

Project 5

Pipeline:

1. Project Setup
2. File Repository
3. Document Extraction
4. Intelligent Chunking
5. Transformer Embeddings
6. Vector Database
7. Hybrid Search
8. Agentic AI
9. Intelligent Recommendation
10. RAG
11. Gradio Interface
12. Evaluation
""",
    encoding="utf-8"
)

print(
    "\n✓ STEP 1 COMPLETED"
)

# ============================================================
# STEP 2 - SMART FILE REPOSITORY
# ============================================================

print("\n")
print("=" * 80)
print("STEP 2 - FILE REPOSITORY / ZIP PROCESSING")
print("=" * 80)

def find_existing_documents():

    documents = []

    if DOCUMENTS_DIR.exists():

        for file_path in DOCUMENTS_DIR.rglob("*"):

            if not file_path.is_file():
                continue

            if (
                file_path.suffix.lower()
                in SUPPORTED_EXTENSIONS
            ):

                documents.append(
                    file_path
                )

    return sorted(
        documents,
        key=lambda x:
        x.name.lower()
    )

document_files = find_existing_documents()

# ------------------------------------------------------------
# CASE 1:
# Documents already exist
# ------------------------------------------------------------

if document_files:

    print(
        "\n✓ Existing document repository detected."
    )

    print(
        "No upload required."
    )

# ------------------------------------------------------------
# CASE 2:
# ZIP already exists in /content
# ------------------------------------------------------------

else:

    zip_candidates = list(
        Path("/content").glob("*.zip")
    )

    if zip_candidates:

        print(
            "\nZIP file detected:"
        )

        for zip_file in zip_candidates:

            print(
                "  ✓",
                zip_file.name
            )

        selected_zip = zip_candidates[0]

        print(
            "\n[ZIP] Processing:",
            selected_zip.name
        )

        extraction_dir = (
            Path("/content")
            /
            "Project5_ZIP_Extracted"
        )

        if extraction_dir.exists():

            shutil.rmtree(
                extraction_dir
            )

        extraction_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        try:

            with zipfile.ZipFile(
                selected_zip,
                "r"
            ) as archive:

                archive.extractall(
                    extraction_dir
                )

            print(
                "[ZIP] Extraction completed."
            )

        except Exception as error:

            print(
                "[ZIP ERROR]",
                error
            )

        extracted_documents = []

        for source_file in extraction_dir.rglob("*"):

            if not source_file.is_file():
                continue

            if (
                source_file.suffix.lower()
                not in SUPPORTED_EXTENSIONS
            ):
                continue

            destination = (
                DOCUMENTS_DIR
                /
                source_file.name
            )

            if destination.exists():

                stem = (
                    source_file.stem
                )

                suffix = (
                    source_file.suffix
                )

                counter = 2

                while destination.exists():

                    destination = (
                        DOCUMENTS_DIR
                        /
                        f"{stem}_{counter}{suffix}"
                    )

                    counter += 1

            shutil.copy2(
                source_file,
                destination
            )

            extracted_documents.append(
                destination
            )

            print(
                "[ZIP → REPOSITORY]",
                destination.name
            )

        print(
            "\nSupported documents extracted:",
            len(extracted_documents)
        )

# ------------------------------------------------------------
# CASE 3:
# Nothing found -> upload ONCE
# ------------------------------------------------------------

document_files = find_existing_documents()

if not document_files:

    print(
        "\n[UPLOAD] No documents found."
    )

    print(
        "Upload the ZIP containing your 5 PDFs."
    )

    print(
        "This upload will happen only once."
    )

    from google.colab import files

    uploaded_files = files.upload()

    for filename, content in uploaded_files.items():

        extension = (
            Path(filename)
            .suffix
            .lower()
        )

        # ----------------------------------------------------
        # ZIP
        # ----------------------------------------------------

        if extension == ".zip":

            print(
                "\n[ZIP] Processing:",
                filename
            )

            zip_path = (
                Path("/content")
                /
                Path(filename).name
            )

            with open(
                zip_path,
                "wb"
            ) as output_file:

                output_file.write(
                    content
                )

            extraction_dir = (
                Path("/content")
                /
                "Project5_ZIP_Extracted"
            )

            if extraction_dir.exists():

                shutil.rmtree(
                    extraction_dir
                )

            extraction_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            with zipfile.ZipFile(
                zip_path,
                "r"
            ) as archive:

                archive.extractall(
                    extraction_dir
                )

            print(
                "[ZIP] Extraction completed."
            )

            for source_file in extraction_dir.rglob("*"):

                if not source_file.is_file():
                    continue

                if (
                    source_file.suffix.lower()
                    not in SUPPORTED_EXTENSIONS
                ):
                    continue

                destination = (
                    DOCUMENTS_DIR
                    /
                    source_file.name
                )

                if destination.exists():

                    stem = source_file.stem
                    suffix = source_file.suffix

                    counter = 2

                    while destination.exists():

                        destination = (
                            DOCUMENTS_DIR
                            /
                            f"{stem}_{counter}{suffix}"
                        )

                        counter += 1

                shutil.copy2(
                    source_file,
                    destination
                )

                print(
                    "[ZIP → REPOSITORY]",
                    destination.name
                )

        # ----------------------------------------------------
        # DIRECT DOCUMENT
        # ----------------------------------------------------

        elif extension in SUPPORTED_EXTENSIONS:

            destination = (
                DOCUMENTS_DIR
                /
                Path(filename).name
            )

            with open(
                destination,
                "wb"
            ) as output_file:

                output_file.write(
                    content
                )

            print(
                "[UPLOADED]",
                destination.name
            )

# ------------------------------------------------------------
# FINAL DOCUMENT CHECK
# ------------------------------------------------------------

document_files = find_existing_documents()

print("\n")
print("-" * 80)

print(
    "DOCUMENT REPOSITORY:"
)

print(
    DOCUMENTS_DIR
)

print(
    "\nSupported documents found:",
    len(document_files)
)

for file_path in document_files:

    print(
        "  ✓",
        file_path.name
    )

print("-" * 80)

if not document_files:

    print(
        "\nNo supported documents are available."
    )

    print(
        "Please restart this cell and upload the ZIP."
    )

    raise SystemExit

print(
    "\n✓ STEP 2 COMPLETED"
)

# ============================================================
# STEP 3 - DOCUMENT EXTRACTION
# ============================================================

print("\n")
print("=" * 80)
print("STEP 3 - DOCUMENT TEXT EXTRACTION")
print("=" * 80)

def extract_pdf(
    file_path
):

    text_parts = []

    try:

        reader = PdfReader(
            str(file_path)
        )

        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            try:

                text = (
                    page.extract_text()
                    or ""
                )

            except Exception:

                text = ""

            if text.strip():

                text_parts.append(
                    f"[Page {page_number}]\n{text}"
                )

    except Exception as error:

        print(
            "[PDF ERROR]",
            file_path.name,
            error
        )

    return "\n\n".join(
        text_parts
    )

def extract_docx(
    file_path
):

    text_parts = []

    try:

        document = Document(
            str(file_path)
        )

        for paragraph in document.paragraphs:

            text = (
                paragraph.text.strip()
            )

            if text:

                text_parts.append(
                    text
                )

        for table in document.tables:

            for row in table.rows:

                row_values = []

                for cell in row.cells:

                    value = (
                        cell.text.strip()
                    )

                    if value:

                        row_values.append(
                            value
                        )

                if row_values:

                    text_parts.append(
                        " | ".join(
                            row_values
                        )
                    )

    except Exception as error:

        print(
            "[DOCX ERROR]",
            file_path.name,
            error
        )

    return "\n".join(
        text_parts
    )

def extract_txt(
    file_path
):

    try:

        return file_path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:

        return ""

def extract_pptx(
    file_path
):

    text_parts = []

    try:

        presentation = Presentation(
            str(file_path)
        )

        for slide_number, slide in enumerate(
            presentation.slides,
            start=1
        ):

            slide_text = []

            for shape in slide.shapes:

                if hasattr(
                    shape,
                    "text"
                ):

                    text = (
                        shape.text.strip()
                    )

                    if text:

                        slide_text.append(
                            text
                        )

            if slide_text:

                text_parts.append(
                    f"[Slide {slide_number}]\n"
                    +
                    "\n".join(
                        slide_text
                    )
                )

    except Exception as error:

        print(
            "[PPTX ERROR]",
            file_path.name,
            error
        )

    return "\n\n".join(
        text_parts
    )

def extract_html(
    file_path
):

    try:

        html = file_path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        for tag in soup(
            [
                "script",
                "style",
                "noscript"
            ]
        ):

            tag.decompose()

        return soup.get_text(
            separator="\n"
        )

    except Exception:

        return ""

def extract_text(
    file_path
):

    extension = (
        file_path.suffix.lower()
    )

    if extension == ".pdf":

        return extract_pdf(
            file_path
        )

    elif extension == ".docx":

        return extract_docx(
            file_path
        )

    elif extension == ".txt":

        return extract_txt(
            file_path
        )

    elif extension == ".pptx":

        return extract_pptx(
            file_path
        )

    elif extension in [
        ".html",
        ".htm"
    ]:

        return extract_html(
            file_path
        )

    return ""

def clean_text(
    text
):

    text = text.replace(
        "\r\n",
        "\n"
    )

    text = text.replace(
        "\r",
        "\n"
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

metadata = []

print(
    "\nExtracting documents..."
)

for file_path in document_files:

    print(
        "\nProcessing:",
        file_path.name
    )

    raw_text = extract_text(
        file_path
    )

    text = clean_text(
        raw_text
    )

    output_file = (
        PROCESSED_DIR
        /
        f"{file_path.stem}.txt"
    )

    output_file.write_text(
        text,
        encoding="utf-8"
    )

    words = text.split()

    metadata.append({

        "filename":
            file_path.name,

        "processed_file":
            output_file.name,

        "characters":
            len(text),

        "words":
            len(words)

    })

    print(
        "Characters:",
        len(text)
    )

    print(
        "Words:",
        len(words)
    )

metadata_file = (
    PROCESSED_DIR
    /
    "metadata.json"
)

metadata_file.write_text(

    json.dumps(
        metadata,
        indent=2
    ),

    encoding="utf-8"
)

total_words = sum(
    item["words"]
    for item in metadata
)

print(
    "\nTotal documents:",
    len(metadata)
)

print(
    "Total extracted words:",
    total_words
)

print(
    "\n✓ STEP 3 COMPLETED"
)

# ============================================================
# STEP 4 - INTELLIGENT CHUNKING
# ============================================================

print("\n")
print("=" * 80)
print("STEP 4 - INTELLIGENT TEXT CHUNKING")
print("=" * 80)

CHUNK_SIZE = 220
CHUNK_OVERLAP = 40

def create_chunks(
    text,
    chunk_size=220,
    overlap=40
):

    words = text.split()

    if not words:

        return []

    chunks = []

    start = 0

    while start < len(words):

        end = min(
            start + chunk_size,
            len(words)
        )

        chunk = " ".join(
            words[start:end]
        )

        if chunk.strip():

            chunks.append(
                chunk
            )

        if end >= len(words):

            break

        start = end - overlap

        if start <= 0:

            start = end

    return chunks

all_chunks = []

processed_files = sorted(
    PROCESSED_DIR.glob("*.txt")
)

for processed_file in processed_files:

    text = processed_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    chunks = create_chunks(
        text,
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    print(
        processed_file.name,
        "→",
        len(chunks),
        "chunks"
    )

    for chunk_number, chunk in enumerate(
        chunks
    ):

        all_chunks.append({

            "chunk_id":
                len(all_chunks),

            "document":
                processed_file.name,

            "chunk_number":
                chunk_number,

            "text":
                chunk

        })

chunks_file = (
    CHUNKS_DIR
    /
    "chunks.json"
)

chunks_file.write_text(

    json.dumps(
        all_chunks,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)

print(
    "\nTotal chunks:",
    len(all_chunks)
)

print(
    "Chunk size:",
    CHUNK_SIZE,
    "words"
)

print(
    "Chunk overlap:",
    CHUNK_OVERLAP,
    "words"
)

print(
    "\n✓ STEP 4 COMPLETED"
)

# ============================================================
# STEP 5 - EMBEDDING GENERATION
# ============================================================

print("\n")
print("=" * 80)
print("STEP 5 - TRANSFORMER EMBEDDING GENERATION")
print("=" * 80)

MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "\nLoading model:"
)

print(
    MODEL_NAME
)

embedding_model = SentenceTransformer(
    MODEL_NAME
)

chunk_texts = [
    item["text"]
    for item in all_chunks
]

print(
    "\nGenerating embeddings..."
)

embeddings = embedding_model.encode(

    chunk_texts,

    batch_size=32,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True

)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

embedding_file = (
    EMBEDDINGS_DIR
    /
    "embeddings.npy"
)

np.save(
    embedding_file,
    embeddings
)

print(
    "\nEmbedding generation completed."
)

print(
    "Shape:",
    embeddings.shape
)

print(
    "Number of vectors:",
    embeddings.shape[0]
)

print(
    "Vector dimension:",
    embeddings.shape[1]
)

print(
    "Saved:",
    embedding_file
)

print(
    "\n✓ STEP 5 COMPLETED"
)

# ============================================================
# STEP 6 - VECTOR DATABASE
# ============================================================

print("\n")
print("=" * 80)
print("STEP 6 - VECTOR DATABASE")
print("=" * 80)

vector_file = (
    VECTOR_DB_DIR
    /
    "vector_store.npy"
)

chunk_index_file = (
    VECTOR_DB_DIR
    /
    "chunk_index.json"
)

np.save(
    vector_file,
    embeddings
)

chunk_index_file.write_text(

    json.dumps(
        all_chunks,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)

FAISS_AVAILABLE = False

try:

    import faiss

    faiss_index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    faiss_index.add(
        embeddings
    )

    faiss_file = (
        VECTOR_DB_DIR
        /
        "faiss.index"
    )

    faiss.write_index(
        faiss_index,
        str(faiss_file)
    )

    FAISS_AVAILABLE = True

    print(
        "✓ FAISS index created."
    )

except Exception:

    print(
        "FAISS not available."
    )

    print(
        "✓ NumPy vector search enabled."
    )

print(
    "\nVector count:",
    len(embeddings)
)

print(
    "Vector dimension:",
    embeddings.shape[1]
)

print(
    "Backend:",
    (
        "FAISS + NumPy"
        if FAISS_AVAILABLE
        else
        "NumPy"
    )
)

print(
    "\n✓ STEP 6 COMPLETED"
)

# ============================================================
# STEP 7 - HYBRID SEARCH
# ============================================================

print("\n")
print("=" * 80)
print("STEP 7 - HYBRID SEMANTIC + KEYWORD SEARCH")
print("=" * 80)

STOPWORDS = {

    "the",
    "is",
    "are",
    "a",
    "an",
    "and",
    "or",
    "of",
    "to",
    "in",
    "on",
    "for",
    "with",
    "what",
    "which",
    "where",
    "how",
    "does",
    "do",
    "about",
    "file",
    "files",
    "document",
    "documents",
    "find",
    "show",
    "me"

}

def tokenize(
    text
):

    words = re.findall(
        r"[a-zA-Z0-9]+",
        text.lower()
    )

    return [

        word

        for word in words

        if (
            len(word) > 2
            and
            word not in STOPWORDS
        )

    ]

def keyword_score(
    query,
    text
):

    query_words = set(
        tokenize(query)
    )

    text_words = set(
        tokenize(text)
    )

    if not query_words:

        return 0.0

    overlap = (
        query_words
        &
        text_words
    )

    return (
        len(overlap)
        /
        len(query_words)
    )

def hybrid_search(
    query,
    top_k=10
):

    query_embedding = (
        embedding_model.encode(

            [query],

            convert_to_numpy=True,

            normalize_embeddings=True

        )[0]
    )

    semantic_scores = np.dot(
        embeddings,
        query_embedding
    )

    results = []

    for index, item in enumerate(
        all_chunks
    ):

        semantic = float(
            semantic_scores[index]
        )

        keyword = keyword_score(
            query,
            item["text"]
        )

        hybrid = (

            0.70 * semantic

            +

            0.30 * keyword

        )

        results.append({

            "chunk_id":
                item["chunk_id"],

            "document":
                item["document"],

            "chunk_number":
                item["chunk_number"],

            "text":
                item["text"],

            "semantic_score":
                semantic,

            "keyword_score":
                keyword,

            "hybrid_score":
                hybrid

        })

    results.sort(

        key=lambda x:
            x["hybrid_score"],

        reverse=True

    )

    return results[:top_k]

test_query = (
    "machine learning artificial intelligence"
)

test_results = hybrid_search(
    test_query,
    top_k=5
)

print(
    "\nTest query:"
)

print(
    test_query
)

print(
    "\nTop search results:"
)

for rank, result in enumerate(
    test_results,
    start=1
):

    print(
        f"\n{rank}. {result['document']}"
    )

    print(
        "   Semantic:",
        f"{result['semantic_score']:.4f}"
    )

    print(
        "   Keyword:",
        f"{result['keyword_score']:.4f}"
    )

    print(
        "   Hybrid:",
        f"{result['hybrid_score']:.4f}"
    )

print(
    "\n✓ STEP 7 COMPLETED"
)

# ============================================================
# STEP 8 - AGENTIC AI
# ============================================================

print("\n")
print("=" * 80)
print("STEP 8 - AGENTIC AI PLANNER + TOOLS")
print("=" * 80)

class FileRecommendationAgent:

    def __init__(self):

        self.trace = []

    def understand_query(
        self,
        query
    ):

        tokens = set(
            tokenize(query)
        )

        domain_keywords = {

            "Artificial Intelligence": {

                "ai",
                "artificial",
                "intelligence",
                "machine",
                "learning",
                "deep",
                "neural"

            },

            "Database": {

                "database",
                "dbms",
                "sql",
                "query",
                "table",
                "normalization"

            },

            "Computer Networks": {

                "network",
                "networks",
                "tcp",
                "udp",
                "routing",
                "dns",
                "protocol",
                "communication"

            },

            "Operating Systems": {

                "operating",
                "process",
                "thread",
                "memory",
                "scheduling",
                "os"

            },

            "Software Engineering": {

                "software",
                "testing",
                "requirements",
                "design",
                "engineering"

            }

        }

        detected_domains = []

        for domain, keywords in (
            domain_keywords.items()
        ):

            if tokens & keywords:

                detected_domains.append(
                    domain
                )

        if not detected_domains:

            detected_domains.append(
                "General Information"
            )

        return {

            "tokens":
                list(tokens),

            "domains":
                detected_domains

        }

    def repository_tool(
        self
    ):

        self.trace.append(
            "TOOL → Repository inspection"
        )

        return sorted(
            set(
                item["document"]
                for item in all_chunks
            )
        )

    def search_tool(
        self,
        query
    ):

        self.trace.append(
            "TOOL → Hybrid semantic search"
        )

        return hybrid_search(
            query,
            top_k=15
        )

    def run(
        self,
        query
    ):

        self.trace = []

        self.trace.append(
            "AGENT → User goal received"
        )

        understanding = (
            self.understand_query(
                query
            )
        )

        self.trace.append(
            "AGENT → Query understanding completed"
        )

        self.trace.append(
            "AGENT → Domain detection completed"
        )

        self.trace.append(
            "AGENT → Search strategy selected"
        )

        search_results = (
            self.search_tool(
                query
            )
        )

        documents = (
            self.repository_tool()
        )

        self.trace.append(
            "AGENT → Relevant chunks collected"
        )

        self.trace.append(
            "AGENT → Files ready for ranking"
        )

        return {

            "query":
                query,

            "understanding":
                understanding,

            "results":
                search_results,

            "documents":
                documents,

            "trace":
                self.trace

        }

agent = FileRecommendationAgent()

agent_test = agent.run(
    "database management systems"
)

print(
    "\nAgent test query:"
)

print(
    agent_test["query"]
)

print(
    "\nDetected domains:"
)

for domain in (
    agent_test[
        "understanding"
    ]["domains"]
):

    print(
        "  ✓",
        domain
    )

print(
    "\nAgent execution trace:"
)

for trace in (
    agent_test["trace"]
):

    print(
        "  →",
        trace
    )

print(
    "\n✓ STEP 8 COMPLETED"
)

# ============================================================
# STEP 9 - INTELLIGENT FILE RECOMMENDATION
# ============================================================

print("\n")
print("=" * 80)
print("STEP 9 - INTELLIGENT FILE RECOMMENDATION")
print("=" * 80)

def recommend_files(
    query,
    top_k=5
):

    search_results = hybrid_search(
        query,
        top_k=25
    )

    grouped = defaultdict(list)

    for result in search_results:

        grouped[
            result["document"]
        ].append(
            result
        )

    recommendations = []

    for filename, items in (
        grouped.items()
    ):

        scores = [

            item["hybrid_score"]

            for item in items

        ]

        best_score = max(
            scores
        )

        average_score = float(
            np.mean(scores)
        )

        chunk_coverage = min(
            len(items) / 3.0,
            1.0
        )

        final_score = (

            0.60 * best_score

            +

            0.25 * average_score

            +

            0.15 * chunk_coverage

        )

        relevance = max(

            0.0,

            min(
                100.0,
                final_score * 100.0
            )

        )

        if relevance >= 65:

            confidence = "HIGH"

        elif relevance >= 45:

            confidence = "MEDIUM"

        else:

            confidence = "LOW"

        best_item = max(

            items,

            key=lambda x:
                x["hybrid_score"]

        )

        recommendations.append({

            "filename":
                filename,

            "relevance":
                round(
                    relevance,
                    2
                ),

            "confidence":
                confidence,

            "matching_chunks":
                len(items),

            "semantic_score":
                round(
                    best_item[
                        "semantic_score"
                    ],
                    4
                ),

            "keyword_score":
                round(
                    best_item[
                        "keyword_score"
                    ],
                    4
                ),

            "evidence":
                best_item[
                    "text"
                ][:700]

        })

    recommendations.sort(

        key=lambda x:
            x["relevance"],

        reverse=True

    )

    return recommendations[
        :top_k
    ]

recommendation_query = (
    "computer networks and communication"
)

recommendations = recommend_files(
    recommendation_query,
    top_k=5
)

print(
    "\nQuery:"
)

print(
    recommendation_query
)

print(
    "\nIntelligent recommendations:"
)

for rank, item in enumerate(
    recommendations,
    start=1
):

    print(
        f"\n{rank}. {item['filename']}"
    )

    print(
        "   Relevance:",
        f"{item['relevance']}%"
    )

    print(
        "   Confidence:",
        item["confidence"]
    )

    print(
        "   Matching chunks:",
        item["matching_chunks"]
    )

    print(
        "   Semantic score:",
        item["semantic_score"]
    )

    print(
        "   Keyword score:",
        item["keyword_score"]
    )

print(
    "\n✓ STEP 9 COMPLETED"
)

# ============================================================
# STEP 10 - RAG + EVIDENCE
# ============================================================

print("\n")
print("=" * 80)
print("STEP 10 - RAG RESPONSE + EVIDENCE")
print("=" * 80)

def generate_rag_response(
    query,
    top_k=5
):

    start_time = time.perf_counter()

    agent_result = agent.run(
        query
    )

    recommendations = recommend_files(
        query,
        top_k
    )

    latency = (
        time.perf_counter()
        -
        start_time
    )

    lines = []

    lines.append(
        "AGENTIC AI FILE RECOMMENDATION RESULT"
    )

    lines.append(
        "=" * 65
    )

    lines.append(
        f"Query: {query}"
    )

    domains = (
        agent_result[
            "understanding"
        ]["domains"]
    )

    lines.append(
        "Detected domain: "
        +
        ", ".join(domains)
    )

    lines.append(
        f"Processing time: "
        f"{latency:.4f} seconds"
    )

    lines.append(
        "\nRECOMMENDED FILES"
    )

    if recommendations:

        for rank, item in enumerate(
            recommendations,
            start=1
        ):

            lines.append(

                f"{rank}. "
                f"{item['filename']} | "
                f"Relevance: "
                f"{item['relevance']}% | "
                f"Confidence: "
                f"{item['confidence']}"

            )

    else:

        lines.append(
            "No relevant files found."
        )

    lines.append(
        "\nSUPPORTING EVIDENCE"
    )

    for rank, item in enumerate(
        recommendations[:3],
        start=1
    ):

        lines.append(
            f"\n[{rank}] "
            f"{item['filename']}"
        )

        lines.append(
            item["evidence"]
        )

    lines.append(
        "\nAGENT EXECUTION TRACE"
    )

    for trace in (
        agent_result["trace"]
    ):

        lines.append(
            "→ " + trace
        )

    answer = "\n".join(
        lines
    )

    result = {

        "query":
            query,

        "answer":
            answer,

        "recommendations":
            recommendations,

        "trace":
            agent_result["trace"],

        "latency":
            latency

    }

    rag_file = (
        RESULTS_DIR
        /
        "rag_result.json"
    )

    rag_file.write_text(

        json.dumps(
            result,
            indent=2,
            ensure_ascii=False
        ),

        encoding="utf-8"
    )

    return result

rag_query = (
    "What files contain information "
    "about computer networks?"
)

rag_result = generate_rag_response(
    rag_query,
    top_k=5
)

print(
    rag_result["answer"]
)

print(
    "\nRAG result saved:"
)

print(
    RESULTS_DIR / "rag_result.json"
)

print(
    "\n✓ STEP 10 COMPLETED"
)

# ============================================================
# STEP 11 - GRADIO USER INTERFACE
# ============================================================

print("\n")
print("=" * 80)
print("STEP 11 - GRADIO USER INTERFACE")
print("=" * 80)

import gradio as gr

def recommendation_interface(
    query
):

    if (
        query is None
        or
        not query.strip()
    ):

        return (

            "Please enter a query.",

            "",

            "",

            ""

        )

    result = generate_rag_response(
        query.strip(),
        top_k=5
    )

    recommendation_lines = []

    for rank, item in enumerate(
        result["recommendations"],
        start=1
    ):

        recommendation_lines.append(

            f"{rank}. "
            f"{item['filename']} | "
            f"{item['relevance']}% | "
            f"{item['confidence']}"

        )

    recommendation_text = (

        "\n".join(
            recommendation_lines
        )

        if recommendation_lines

        else

        "No relevant files found."

    )

    trace_text = "\n".join(

        "→ " + trace

        for trace in result[
            "trace"
        ]

    )

    statistics = (

        f"Documents: "
        f"{len(document_files)}\n"

        f"Chunks: "
        f"{len(all_chunks)}\n"

        f"Embedding dimension: "
        f"{embeddings.shape[1]}\n"

        f"Vector backend: "
        f"{'FAISS + NumPy' if FAISS_AVAILABLE else 'NumPy'}\n"

        f"Search method: "
        f"Hybrid semantic + keyword\n"

        f"Processing time: "
        f"{result['latency']:.4f} seconds"

    )

    return (

        recommendation_text,

        result["answer"],

        trace_text,

        statistics

    )

app = gr.Blocks(
    title=
    "Agentic AI File Recommendation"
)

with app:

    gr.Markdown(
        """
# 🤖 Agentic AI-Based Intelligent File Recommendation System

### Project 5

**Query → Understand → Plan → Search → Rank → Recommend → Explain**

Enter a question and the agent will search the document repository,
rank the most relevant files, and provide supporting evidence.
"""
    )

    query_box = gr.Textbox(

        label="Enter your query",

        placeholder=(
            "Example: Which files contain "
            "information about database management?"
        ),

        lines=3

    )

    search_button = gr.Button(
        "🔍 Recommend Files"
    )

    recommended_box = gr.Textbox(

        label="Recommended Files",

        lines=8

    )

    explanation_box = gr.Textbox(

        label="AI Explanation + Evidence",

        lines=18

    )

    trace_box = gr.Textbox(

        label="Agent Execution Trace",

        lines=12

    )

    statistics_box = gr.Textbox(

        label="System Statistics",

        lines=8

    )

    search_button.click(

        fn=recommendation_interface,

        inputs=query_box,

        outputs=[

            recommended_box,

            explanation_box,

            trace_box,

            statistics_box

        ]

    )

print(
    "\n✓ Gradio interface created."
)

print(
    "\n✓ STEP 11 COMPLETED"
)

# ============================================================
# STEP 12 - TESTING + EVALUATION
# ============================================================

print("\n")
print("=" * 80)
print("STEP 12 - TESTING + EVALUATION")
print("=" * 80)

test_queries = [

    "machine learning and artificial intelligence",

    "database management systems",

    "computer networks and communication",

    "operating systems process management",

    "software engineering"

]

evaluation_results = []

for test_number, query in enumerate(
    test_queries,
    start=1
):

    print(
        "\n"
        +
        "-" * 75
    )

    print(
        f"TEST {test_number}"
    )

    print(
        "Query:",
        query
    )

    start_time = time.perf_counter()

    results = recommend_files(
        query,
        top_k=3
    )

    latency = (
        time.perf_counter()
        -
        start_time
    )

    if results:

        top = results[0]

        print(
            "Top recommendation:",
            top["filename"]
        )

        print(
            "Relevance:",
            f"{top['relevance']}%"
        )

        print(
            "Confidence:",
            top["confidence"]
        )

        top_file = (
            top["filename"]
        )

        top_score = (
            top["relevance"]
        )

        confidence = (
            top["confidence"]
        )

    else:

        print(
            "No recommendation found."
        )

        top_file = "None"

        top_score = 0

        confidence = "NONE"

    print(
        "Latency:",
        f"{latency:.4f} seconds"
    )

    evaluation_results.append({

        "test_number":
            test_number,

        "query":
            query,

        "top_file":
            top_file,

        "top_score":
            top_score,

        "confidence":
            confidence,

        "latency_seconds":
            round(
                latency,
                4
            )

    })

average_latency = float(
    np.mean([

        item["latency_seconds"]

        for item in evaluation_results

    ])
)

evaluation_dashboard = {

    "project":
        "Agentic AI-Based Intelligent File Recommendation System",

    "documents":
        len(document_files),

    "chunks":
        len(all_chunks),

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        int(
            embeddings.shape[1]
        ),

    "vector_backend":
        (
            "FAISS + NumPy"
            if FAISS_AVAILABLE
            else
            "NumPy"
        ),

    "search_method":
        "Hybrid Semantic + Keyword",

    "average_latency_seconds":
        round(
            average_latency,
            4
        ),

    "tests":
        evaluation_results

}

evaluation_file = (
    RESULTS_DIR
    /
    "evaluation_dashboard.json"
)

evaluation_file.write_text(

    json.dumps(
        evaluation_dashboard,
        indent=2,
        ensure_ascii=False
    ),

    encoding="utf-8"
)

print(
    "\nEvaluation dashboard saved:"
)

print(
    evaluation_file
)

print(
    "\n✓ STEP 12 COMPLETED"
)

# ============================================================
# FINAL PROJECT DASHBOARD
# ============================================================

print("\n")
print("=" * 80)
print("PROJECT 5 - FINAL SYSTEM DASHBOARD")
print("=" * 80)

print(
    "\nDOCUMENT INTELLIGENCE"
)

print(
    "Documents:",
    len(document_files)
)

print(
    "Total chunks:",
    len(all_chunks)
)

print(
    "\nEMBEDDING MODEL"
)

print(
    MODEL_NAME
)

print(
    "Dimension:",
    embeddings.shape[1]
)

print(
    "\nSEARCH ENGINE"
)

print(
    "Hybrid semantic + keyword search"
)

print(
    "Vector backend:",
    (
        "FAISS + NumPy"
        if FAISS_AVAILABLE
        else
        "NumPy"
    )
)

print(
    "\nAGENTIC AI"
)

print(
    "✓ Query understanding"
)

print(
    "✓ Domain detection"
)

print(
    "✓ Planning"
)

print(
    "✓ Tool execution"
)

print(
    "✓ Search"
)

print(
    "✓ File ranking"
)

print(
    "✓ Confidence scoring"
)

print(
    "\nRAG"
)

print(
    "✓ Evidence retrieval"
)

print(
    "✓ Supporting evidence"
)

print(
    "\nUSER INTERFACE"
)

print(
    "✓ Gradio interface ready"
)

print(
    "\nEVALUATION"
)

print(
    "Test queries:",
    len(evaluation_results)
)

print(
    "Average latency:",
    f"{average_latency:.4f} seconds"
)

print(
    "\nOUTPUT FILES"
)

print(
    "Processed documents:",
    PROCESSED_DIR
)

print(
    "Chunks:",
    CHUNKS_DIR
)

print(
    "Embeddings:",
    EMBEDDINGS_DIR
)

print(
    "Vector database:",
    VECTOR_DB_DIR
)

print(
    "Results:",
    RESULTS_DIR
)

print("\n")
print("=" * 80)

print(
    "🎉 ALL 12 STEPS COMPLETED SUCCESSFULLY"
)

print("=" * 80)

print(
    """
✓ STEP 1  - Project Setup
✓ STEP 2  - File Repository
✓ STEP 3  - Document Extraction
✓ STEP 4  - Intelligent Chunking
✓ STEP 5  - Transformer Embeddings
✓ STEP 6  - Vector Database
✓ STEP 7  - Hybrid Semantic Search
✓ STEP 8  - Agentic AI
✓ STEP 9  - Intelligent File Recommendation
✓ STEP 10 - RAG + Evidence
✓ STEP 11 - Gradio Interface
✓ STEP 12 - Testing + Evaluation
"""
)

print("=" * 80)

# ============================================================
# OPTIONAL UI LAUNCH
# ============================================================

# Change False to True if you want the Gradio interface
# to launch automatically after all 12 steps.

LAUNCH_UI = False

if LAUNCH_UI:

    print(
        "\nLaunching Gradio interface..."
    )

    app.launch(
        share=True,
        debug=False
    )

else:

    print(
        "\nGradio interface is ready."
    )

    print(
        "To launch it, run:"
    )

    print(
        "app.launch(share=True)"
    )

print(
    "\nPROJECT 5 COMPLETE."
)


PROJECT 5
AGENTIC AI-BASED INTELLIGENT FILE RECOMMENDATION SYSTEM
COMPLETE EXECUTION


SYSTEM SETUP
✓ numpy already installed
Installing pypdf...
✓ pypdf installed
Installing python-docx...
✓ python-docx installed
Installing python-pptx...
✓ python-pptx installed
✓ sentence-transformers already installed
✓ gradio already installed
✓ beautifulsoup4 already installed


STEP 1 - PROJECT SETUP

Project directory:
/content/Agentic_File_Recommendation

Created project folders:
  ✓ /content/Agentic_File_Recommendation/data/documents
  ✓ /content/Agentic_File_Recommendation/data/processed
  ✓ /content/Agentic_File_Recommendation/data/chunks
  ✓ /content/Agentic_File_Recommendation/embeddings
  ✓ /content/Agentic_File_Recommendation/vector_db
  ✓ /content/Agentic_File_Recommendation/results
  ✓ /content/Agentic_File_Recommendation/app
  ✓ /content/Agentic_File_Recommendation/src

✓ STEP 1 COMPLETED


STEP 2 - FILE REPOSITORY / ZIP PROCESSING

[UPLOAD] No documents found.
Upload the ZIP containi

Saving Agentic_File_Recommendation_Website_Package (1).zip to Agentic_File_Recommendation_Website_Package (1).zip

[ZIP] Processing: Agentic_File_Recommendation_Website_Package (1).zip
[ZIP] Extraction completed.
[ZIP → REPOSITORY] DEPLOYMENT_GUIDE.txt
[ZIP → REPOSITORY] requirements.txt
[ZIP → REPOSITORY] Software_Engineering.pdf
[ZIP → REPOSITORY] OS.pdf
[ZIP → REPOSITORY] DBMS.pdf
[ZIP → REPOSITORY] AI.pdf
[ZIP → REPOSITORY] Networks.pdf


--------------------------------------------------------------------------------
DOCUMENT REPOSITORY:
/content/Agentic_File_Recommendation/data/documents

Supported documents found: 7
  ✓ AI.pdf
  ✓ DBMS.pdf
  ✓ DEPLOYMENT_GUIDE.txt
  ✓ Networks.pdf
  ✓ OS.pdf
  ✓ requirements.txt
  ✓ Software_Engineering.pdf
--------------------------------------------------------------------------------

✓ STEP 2 COMPLETED


STEP 3 - DOCUMENT TEXT EXTRACTION

Extracting documents...

Processing: AI.pdf
Characters: 580
Words: 71

Processing: DBMS.pdf
Characters: 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding generation completed.
Shape: (7, 384)
Number of vectors: 7
Vector dimension: 384
Saved: /content/Agentic_File_Recommendation/embeddings/embeddings.npy

✓ STEP 5 COMPLETED


STEP 6 - VECTOR DATABASE
FAISS not available.
✓ NumPy vector search enabled.

Vector count: 7
Vector dimension: 384
Backend: NumPy

✓ STEP 6 COMPLETED


STEP 7 - HYBRID SEMANTIC + KEYWORD SEARCH

Test query:
machine learning artificial intelligence

Top search results:

1. AI.txt
   Semantic: 0.5826
   Keyword: 1.0000
   Hybrid: 0.7078

2. DBMS.txt
   Semantic: 0.1296
   Keyword: 0.0000
   Hybrid: 0.0907

3. Networks.txt
   Semantic: 0.1059
   Keyword: 0.0000
   Hybrid: 0.0741

4. OS.txt
   Semantic: 0.0810
   Keyword: 0.0000
   Hybrid: 0.0567

5. DEPLOYMENT_GUIDE.txt
   Semantic: 0.0768
   Keyword: 0.0000
   Hybrid: 0.0538

✓ STEP 7 COMPLETED


STEP 8 - AGENTIC AI PLANNER + TOOLS

Agent test query:
database management systems

Detected domains:
  ✓ Database

Agent execution trace:
  → AGENT → User goal r

In [2]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d347528fcb13d4f006.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

# ============================================================
# PROJECT 5: AGENTIC AI FILE RECOMMENDATION WEBSITE
# COMPLETE GOOGLE COLAB CODE
# Includes 10 test queries, semantic search, evidence, and UI
# ============================================================

%pip -q install gradio numpy pypdf sentence-transformers

import os
import re
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# 1. FIND YOUR EXISTING PROJECT DOCUMENTS
# ------------------------------------------------------------

possible_folders = [
    Path("/content/Agentic_File_Recommendation/data/documents"),
    Path("/content/data/documents"),
    Path("/content/Agentic_File_Recommendation/documents"),
    Path("/content/documents"),
]

DOC_DIR = next(
    (folder for folder in possible_folders
     if folder.exists()
     and any(folder.rglob("*.pdf"))),
    Path("/content/Agentic_File_Recommendation/data/documents")
)

DOC_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path("/content/Agentic_File_Recommendation/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(DOC_DIR.rglob("*.pdf"))

print("=" * 65)
print("AGENTIC AI FILE RECOMMENDATION WEBSITE")
print("=" * 65)
print("Document folder:", DOC_DIR)
print("PDF documents found:", len(pdf_files))

if not pdf_files:
    raise RuntimeError(
        "No PDF documents were found. Upload your PDFs to "
        f"{DOC_DIR}, then run this cell again."
    )

for pdf in pdf_files:
    print("  [FOUND]", pdf.name)

# ------------------------------------------------------------
# 2. EXTRACT AND CHUNK DOCUMENT TEXT
# ------------------------------------------------------------

CHUNK_SIZE = 220
CHUNK_OVERLAP = 40
all_chunks = []

def extract_pdf_text(path):
    try:
        reader = PdfReader(str(path))
        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )
    except Exception as exc:
        print("[WARNING] Could not read", path.name, exc)
        return ""

def make_chunks(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()

    if not words:
        return []

    step = max(1, size - overlap)
    output = []

    for start in range(0, len(words), step):
        part = words[start:start + size]
        if part:
            output.append(" ".join(part))
        if start + size >= len(words):
            break

    return output

print("\nExtracting documents...")

for path in pdf_files:
    text = extract_pdf_text(path)
    chunks = make_chunks(text)

    for number, chunk in enumerate(chunks, 1):
        all_chunks.append({
            "file_name": path.name,
            "chunk_number": number,
            "text": chunk
        })

    print(f"[PROCESSED] {path.name}: {len(chunks)} chunks")

if not all_chunks:
    raise RuntimeError(
        "No readable PDF text was found. "
        "Scanned/image-only PDFs may require OCR."
    )

# ------------------------------------------------------------
# 3. GENERATE TRANSFORMER EMBEDDINGS
# ------------------------------------------------------------

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("\nLoading embedding model:", MODEL_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)

print("Generating embeddings...")

embeddings = embedding_model.encode(
    [item["text"] for item in all_chunks],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)

np.save(RESULTS_DIR / "website_embeddings.npy", embeddings)

print("Embedding shape:", embeddings.shape)
print("Total chunks:", len(all_chunks))

# ------------------------------------------------------------
# 4. SEMANTIC SEARCH AND FILE RANKING
# ------------------------------------------------------------

def semantic_search(query, top_k=15):
    query_vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].astype(np.float32)

    scores = embeddings @ query_vector
    indices = np.argsort(scores)[::-1][:min(top_k, len(scores))]

    results = []

    for index in indices:
        item = dict(all_chunks[int(index)])
        item["score"] = float(scores[int(index)])
        results.append(item)

    return results

def recommend_files(query):
    query = (query or "").strip()

    if not query:
        return (
            "Please enter a query or select an example.",
            "",
            "No query submitted.",
            get_statistics()
        )

    retrieved = semantic_search(query, top_k=20)
    grouped = defaultdict(list)

    for item in retrieved:
        grouped[item["file_name"]].append(item)

    ranked = []

    for filename, passages in grouped.items():
        scores = [p["score"] for p in passages]
        best = max(scores)
        average = sum(scores) / len(scores)

        final_score = 0.70 * best + 0.30 * average

        ranked.append({
            "file_name": filename,
            "score": final_score,
            "passages": sorted(
                passages,
                key=lambda p: p["score"],
                reverse=True
            )
        })

    ranked.sort(key=lambda item: item["score"], reverse=True)
    ranked = ranked[:5]

    # Recommended file list
    result_lines = ["RECOMMENDED FILES"]

    for rank, item in enumerate(ranked, 1):
        result_lines.append(
            f"{rank}. {item['file_name']} | "
            f"Relevance score: {item['score']:.4f}"
        )

    # Evidence from actual retrieved document text
    evidence_lines = [
        f"User query: {query}",
        "",
        "RETRIEVED EVIDENCE"
    ]

    for rank, item in enumerate(ranked, 1):
        evidence_lines.append(
            f"\n{rank}. {item['file_name']}"
        )

        for passage in item["passages"][:2]:
            evidence_lines.append(
                f"Chunk {passage['chunk_number']} "
                f"(similarity {passage['score']:.4f}):\n"
                f"{passage['text'][:700]}"
            )

    # Transparent agent workflow trace
    trace = "\n".join([
        "AGENT EXECUTION TRACE",
        "1. Received the tester's query.",
        "2. Encoded the query with a transformer model.",
        "3. Compared it with indexed document chunks.",
        "4. Retrieved the most similar passages.",
        "5. Grouped passages by filename.",
        "6. Ranked files using maximum and average scores.",
        "7. Prepared evidence from retrieved passages.",
        "8. Returned the recommendations."
    ])

    # Save latest query results
    with open(
        RESULTS_DIR / "latest_test_result.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump({
            "query": query,
            "recommendations": [
                {
                    "file_name": item["file_name"],
                    "score": item["score"]
                }
                for item in ranked
            ]
        }, file, indent=2, ensure_ascii=False)

    return (
        "\n".join(result_lines),
        "\n".join(evidence_lines),
        trace,
        get_statistics()
    )

# ------------------------------------------------------------
# 5. SYSTEM STATISTICS
# ------------------------------------------------------------

def get_statistics():
    return (
        f"PDF documents: {len(pdf_files)}\n"
        f"Indexed text chunks: {len(all_chunks)}\n"
        f"Embedding dimensions: {embeddings.shape[1]}\n"
        f"Embedding model: {MODEL_NAME}\n"
        f"Search method: Cosine similarity\n"
        f"Ranking: 70% best match + 30% average match\n"
        f"Supported files in this version: PDF"
    )

# ------------------------------------------------------------
# 6. TEN TEST QUERIES
# ------------------------------------------------------------

test_queries = [
    ["Which files contain information about database management?"],
    ["Which files contain information about operating systems and process management?"],
    ["Which files discuss computer networks and routing?"],
    ["Which documents explain artificial intelligence and machine learning?"],
    ["Which files contain information about software engineering and software testing?"],
    ["Find documents related to database normalization and SQL."],
    ["Which files explain memory management and CPU scheduling?"],
    ["Find documents about network protocols and network security."],
    ["Which files discuss deep learning and AI applications?"],
    ["Find documents that explain software development life cycle and requirements engineering."]
]

# ------------------------------------------------------------
# 7. CREATE WEBSITE INTERFACE
# ------------------------------------------------------------

with gr.Blocks(
    title="Agentic AI File Recommendation System",
    theme=gr.themes.Soft()
) as app:

    gr.Markdown("""
# 🤖 Agentic AI-Based Intelligent File Recommendation System

### Project 5

**Query → Understand → Search → Rank → Recommend → Explain**

Enter a question to find relevant documents, inspect evidence,
and view the agent's execution trace.
""")

    query_box = gr.Textbox(
        label="Enter your query",
        placeholder="Example: Which files contain information about database management?",
        lines=3
    )

    gr.Examples(
        examples=test_queries,
        inputs=query_box,
        label="🧪 Choose from 10 Test Queries"
    )

    search_button = gr.Button(
        "🔍 Recommend Files",
        variant="primary"
    )

    with gr.Tab("Recommended Files"):
        file_output = gr.Textbox(
            label="Ranked Recommendations",
            lines=10
        )

    with gr.Tab("AI Explanation + Evidence"):
        evidence_output = gr.Textbox(
            label="Evidence from Documents",
            lines=20
        )

    with gr.Tab("Agent Execution Trace"):
        trace_output = gr.Textbox(
            label="Agent Workflow",
            lines=12
        )

    with gr.Tab("System Statistics"):
        statistics_output = gr.Textbox(
            label="Repository Statistics",
            value=get_statistics(),
            lines=8
        )

    outputs = [
        file_output,
        evidence_output,
        trace_output,
        statistics_output
    ]

    search_button.click(
        fn=recommend_files,
        inputs=query_box,
        outputs=outputs
    )

    query_box.submit(
        fn=recommend_files,
        inputs=query_box,
        outputs=outputs
    )

    gr.Markdown("""
---
**Tester instructions:** Select one of the 10 example queries,
click **Recommend Files**, and inspect all four result sections.

Relevance scores are similarity-based ranking scores, not guaranteed
accuracy percentages. Results depend on the indexed document contents.
""")

print("\nWebsite interface created successfully.")

# ------------------------------------------------------------
# 8. LAUNCH WEBSITE IN COLAB
# ------------------------------------------------------------

print("\nLaunching the website...")
print("Wait for Gradio to generate the public URL.")

app.launch(
    share=True,
    debug=True,
    inbrowser=False
)


AGENTIC AI FILE RECOMMENDATION WEBSITE
Document folder: /content/Agentic_File_Recommendation/data/documents
PDF documents found: 5
  [FOUND] AI.pdf
  [FOUND] DBMS.pdf
  [FOUND] Networks.pdf
  [FOUND] OS.pdf
  [FOUND] Software_Engineering.pdf

Extracting documents...
[PROCESSED] AI.pdf: 1 chunks
[PROCESSED] DBMS.pdf: 1 chunks
[PROCESSED] Networks.pdf: 1 chunks
[PROCESSED] OS.pdf: 1 chunks
[PROCESSED] Software_Engineering.pdf: 1 chunks

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (5, 384)
Total chunks: 5

Website interface created successfully.

Launching the website...
Wait for Gradio to generate the public URL.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6e5e95f343ae804e38.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
